In [1]:
!pip uninstall -y transformers
!pip install transformers==4.39.3 tf-keras datasets scikit-learn pandas matplotlib

Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0

 Starting DistilBERT Fine-Tuning Pipeline (COLAB GPU)...
Loading Cleaned Dataset...


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


 Initializing DistilBERT Tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

 Tokenizing Datasets (This takes a moment)...
 Loading Pre-trained TFDistilBertForSequenceClassification...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 


⚡ Commencing DistilBERT Fine-Tuning on GPU...

Epoch 1/3
719/719 [==============================] - 326s 420ms/step - loss: 0.3584 - accuracy: 0.8280 - val_loss: 0.3026 - val_accuracy: 0.8611 - lr: 3.0000e-05
Epoch 2/3
719/719 [==============================] - 305s 425ms/step - loss: 0.2680 - accuracy: 0.8747 - val_loss: 0.2880 - val_accuracy: 0.8670 - lr: 3.0000e-05
Epoch 3/3
719/719 [==============================] - ETA: 0s - loss: 0.2325 - accuracy: 0.8949
Epoch 3: ReduceLROnPlateau reducing learning rate to 1.4999999621068127e-05.
719/719 [==============================] - 305s 424ms/step - loss: 0.2325 - accuracy: 0.8949 - val_loss: 0.3053 - val_accuracy: 0.8579 - lr: 3.0000e-05
Restoring model weights from the end of the best epoch: 2.

 Evaluating Final Model on Test Set...
90/90 [==============================] - 13s 146ms/step - loss: 0.2759 - accuracy: 0.8674
****************************************
 Final Test Accuracy: 0.8674
****************************************
 Bes

In [3]:
!cd /content/models && zip -r best_distilbert.zip best_distilbert

/bin/bash: line 1: cd: /content/models: No such file or directory


In [3]:
# =======================================================
# DistilBERT Fine-Tuning Script (COLAB GPU) + AUTO-ZIP
# =======================================================
import os
import shutil # ZIP ke liye
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["USE_TF"] = "1"

import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from transformers import DistilBertTokenizerFast, TFDistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

np.random.seed(42)
tf.random.set_seed(42)

CLEAN_DIR = "/content/clean"
ARTIFACTS_DIR = "/content/artifacts"
MODELS_DIR = "/content/models"
os.makedirs(MODELS_DIR, exist_ok=True)

def plot_training_history(history, models_dir):
    print("\n Generating Training History Plot...")
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy (DistilBERT)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss (DistilBERT)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plot_path = os.path.join(models_dir, "training_history_distilbert.png")
    plt.tight_layout()
    plt.savefig(plot_path, dpi=300)
    plt.close()
    print(f" Training plot saved at: {plot_path}")

def train_distilbert():
    print(" Starting DistilBERT Fine-Tuning Pipeline (COLAB GPU)...")

    clean_csv_path = os.path.join(CLEAN_DIR, "cleaned_programming_problems.csv")

    print(" Loading Cleaned Dataset...")
    df = pd.read_csv(clean_csv_path)
    df = df.dropna(subset=['model_text', 'difficulty'])
    df = df[df['difficulty'].astype(str).str.lower() != 'unknown']

    texts = df['model_text'].astype(str).tolist()
    labels = df['difficulty'].values

    print(f" Total Questions for Training: {len(texts)}") # Yeh check karega ki poora data aaya ya nahi

    encoder_path = os.path.join(ARTIFACTS_DIR, "difficulty_encoder.pkl")
    with open(encoder_path, 'rb') as f:
        encoder = pickle.load(f)

    encoded_labels = encoder.transform(labels)
    num_classes = len(encoder.classes_)

    X_train, X_temp, y_train, y_temp = train_test_split(
        texts, encoded_labels, test_size=0.20, random_state=42, stratify=encoded_labels
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )

    print(" Initializing DistilBERT Tokenizer...")
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    max_len = 128

    print("Tokenizing Datasets (This takes a moment)...")
    train_encodings = tokenizer(X_train, truncation=True, padding="max_length", max_length=max_len, return_tensors="tf")
    val_encodings = tokenizer(X_val, truncation=True, padding="max_length", max_length=max_len, return_tensors="tf")
    test_encodings = tokenizer(X_test, truncation=True, padding="max_length", max_length=max_len, return_tensors="tf")

    train_x = dict(train_encodings)
    val_x = dict(val_encodings)
    test_x = dict(test_encodings)

    print(" Loading Pre-trained TFDistilBertForSequenceClassification...")
    model = TFDistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased',
        num_labels=num_classes
    )

    optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

    model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=2, restore_best_weights=True, verbose=1
    )
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6, verbose=1
    )

    print("\n⚡ Commencing DistilBERT Fine-Tuning on GPU...\n")

    history = model.fit(
        x=train_x,
        y=y_train,
        validation_data=(val_x, y_val),
        batch_size=32,
        epochs=3,
        callbacks=[early_stopping, reduce_lr]
    )

    print("\n Evaluating Final Model on Test Set...")
    test_loss, test_acc = model.evaluate(test_x, y_test, batch_size=32, verbose=1)
    print("*"*40)
    print(f" Final Test Accuracy: {test_acc:.4f}")
    print("*"*40)

    best_model_dir = os.path.join(MODELS_DIR, "best_distilbert")
    model.save_pretrained(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)
    print(f" Best DistilBERT model & tokenizer saved at: {best_model_dir}")

    with open(os.path.join(MODELS_DIR, "training_history_distilbert.json"), "w") as f:
        history_dict = {k: [float(val) for val in v] for k, v in history.history.items()}
        json.dump(history_dict, f, indent=4)

    plot_training_history(history, MODELS_DIR)

    #  YAHAN AUTO-ZIP HOGA
    print("\n Zipping the model automatically...")
    shutil.make_archive("/content/best_distilbert_model", 'zip', best_model_dir)
    print(" DONE! Download 'best_distilbert_model.zip' from the left panel!")

if __name__ == "__main__":
    train_distilbert()

 Starting DistilBERT Fine-Tuning Pipeline (COLAB GPU)...
 Loading Cleaned Dataset...


/tmp/ipykernel_1143/3321770292.py:56: DtypeWarning: Columns (6,7,10,11,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(clean_csv_path)
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


 Total Questions for Training: 72640
 Initializing DistilBERT Tokenizer...
Tokenizing Datasets (This takes a moment)...
 Loading Pre-trained TFDistilBertForSequenceClassification...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 


⚡ Commencing DistilBERT Fine-Tuning on GPU...

Epoch 1/3
1816/1816 [==============================] - 799s 431ms/step - loss: 0.3013 - accuracy: 0.8575 - val_loss: 0.2396 - val_accuracy: 0.8838 - lr: 3.0000e-05
Epoch 2/3
1816/1816 [==============================] - 777s 428ms/step - loss: 0.2322 - accuracy: 0.8917 - val_loss: 0.2389 - val_accuracy: 0.8850 - lr: 3.0000e-05
Epoch 3/3
1816/1816 [==============================] - ETA: 0s - loss: 0.1990 - accuracy: 0.9087
Epoch 3: ReduceLROnPlateau reducing learning rate to 1.4999999621068127e-05.
1816/1816 [==============================] - 776s 427ms/step - loss: 0.1990 - accuracy: 0.9087 - val_loss: 0.2394 - val_accuracy: 0.8910 - lr: 3.0000e-05
Restoring model weights from the end of the best epoch: 2.

 Evaluating Final Model on Test Set...
227/227 [==============================] - 33s 145ms/step - loss: 0.2431 - accuracy: 0.8827
****************************************
 Final Test Accuracy: 0.8827
***********************************